# Module 13 Lab: Deploying a Computer Vision Model as an API

Welcome to the final and most practical lab of the course! So far, you have learned how to build and train computer vision models. But how do you make your model useful to others? In this lab, you will learn how to **deploy** a model as an **API (Application Programming Interface)**. This will allow other applications to use your model to make predictions.

**What you'll learn:**
- What it means to deploy a model.
- What an API is and why it's useful.
- How to use the FastAPI framework to create a simple API for your model.
- How to send a request to your API and get a prediction back.

## Learning Objectives

By the end of this lab, you will be able to:

- **Explain** the concept of model deployment and its importance.
- **Create** a simple web API using FastAPI.
- **Integrate** a pre-trained computer vision model into a FastAPI application.
- **Deploy** a computer vision model as a local API.

## 1. Setup and Installation

We will need a few new libraries for this lab, including `fastapi` for creating the API and `uvicorn` for running it.


In [15]:
!pip install fastapi uvicorn python-multipart transformers torch

## 2. Understanding Model Deployment and APIs

### What is Model Deployment?

**Model deployment** is the process of taking your trained model and making it available for use in a production environment. This means that other people and applications can send data to your model and get predictions back. It's the final step in the machine learning lifecycle.

### What is an API?

An **API (Application Programming Interface)** is a set of rules and protocols that allows different software applications to communicate with each other. In our case, we will create a web API that allows other applications to communicate with our model over the internet using standard HTTP requests.

### Why use FastAPI?

**FastAPI** is a modern, fast (high-performance) web framework for building APIs with Python. It is very easy to learn and use, and it automatically generates interactive API documentation, which is a huge plus!

## Part 1: Coded Demonstration

In this part, we will create a simple API that can classify an image using a pre-trained model from Hugging Face. Because we are in a notebook environment, we will write the code to a Python file and then run it.

In [18]:
# 1. Write the API code to a Python file

api_code = '''
from fastapi import FastAPI, File, UploadFile
from transformers import pipeline
from PIL import Image
import io

# Create the FastAPI app
app = FastAPI()

# Load the image classification pipeline
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

@app.get("/")
def read_root():
    return {"message": "Welcome to the Image Classification API!"}

@app.post("/classify/")
async def classify_image(file: UploadFile = File(...)):
    # Read the image file
    contents = await file.read()
    image = Image.open(io.BytesIO(contents))

    # Get the prediction
    prediction = classifier(image)

    return {"filename": file.filename, "prediction": prediction}
'''

with open("main.py", "w") as f:
    f.write(api_code)

### 2. Run the API

Now, we will run the API using `uvicorn`. This will start a local web server that listens for requests. We will run this in the background so we can continue to use the notebook.

**Note:** You may need to stop and restart the kernel after running this cell to run the client code in the next step.

In [19]:
import subprocess
import time

# Start the server in the background
server_process = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])

# Wait a moment for the server to start
time.sleep(5)

### 3. Test the API

Now that the API is running, let's send it an image and see what it predicts!

In [27]:
import requests
from PIL import Image
import io

# Download an image to test
image_url = "https://images.unsplash.com/photo-1543466835-00a7907e9de1?ixlib=rb-4.0.3&ixid=MnwxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8&auto=format&fit=crop&w=1074&q=80"
response = requests.get(image_url)
image_bytes = io.BytesIO(response.content)

# Send the image to the API
files = {"file": ("dog.jpg", image_bytes, "image/jpeg")}
response = requests.post("http://localhost:8000/classify/", files=files)

# Print the prediction
print(response.json())

{'filename': 'dog.jpg', 'prediction': [{'label': 'beagle', 'score': 0.6580894589424133}, {'label': 'English foxhound', 'score': 0.26389145851135254}, {'label': 'Walker hound, Walker foxhound', 'score': 0.04986322671175003}, {'label': 'Brittany spaniel', 'score': 0.0036589631345123053}, {'label': 'EntleBucher', 'score': 0.0034517841413617134}]}


### 4. Stop the API Server

It's important to stop the server process when you're done.

In [ ]:
server_process.terminate()

## Part 2: Student Challenge

Your challenge is to create a new API that uses the YOLO object detection model from a previous lab. This will require you to modify the API to handle object detection instead of image classification.

**Your Task:**
1.  Create a new Python file called `object_detection_api.py`.
2.  In this file, create a FastAPI application that uses the `yolov8n.pt` model to detect objects in an image.
3.  The API should have an endpoint that accepts an image and returns a list of detected objects with their bounding boxes.
4.  Run your new API and test it with an image.

In [40]:
!pip install fastapi uvicorn python-multipart transformers torch ultralytics pillow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.2 MB/s eta 0:00:00


In [41]:

# --- ENTER YOUR CODE HERE ---

# 1. Write your object detection API code to a file
# ...

# 2. Run your API
# ...

# 3. Test your API
# ...

# 4. Stop your API
# ...


# ------------------------------------------------------------
# 1. Write your object detection API code to a file
# ------------------------------------------------------------

object_detection_api_code = '''
from fastapi import FastAPI, File, UploadFile
from ultralytics import YOLO
from PIL import Image
import io

app = FastAPI()

# Load YOLOv8 nano model once at startup
model = YOLO("yolov8n.pt")

@app.get("/")
def read_root():
    return {"message": "Welcome to the YOLO Object Detection API!"}

@app.post("/detect/")
async def detect_objects(file: UploadFile = File(...)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    # Run YOLO inference
    results = model.predict(image, verbose=False)
    result = results[0]

    detections = []
    if result.boxes is not None:
        for box in result.boxes:
            class_id = int(box.cls)
            confidence = float(box.conf)
            x1, y1, x2, y2 = box.xyxy.tolist()[0]

            detections.append({
                "label": model.names[class_id],
                "confidence": round(confidence, 4),
                "bounding_box": {
                    "x1": round(x1, 2),
                    "y1": round(y1, 2),
                    "x2": round(x2, 2),
                    "y2": round(y2, 2)
                }
            })

    return {"filename": file.filename, "detections": detections}
'''

with open("object_detection_api.py", "w") as f:
    f.write(object_detection_api_code)

print("Created object_detection_api.py for Mary Ann")

# ------------------------------------------------------------
# 2. Run your API (Colab-safe with logs)
# ------------------------------------------------------------

!nohup uvicorn object_detection_api:app --host 0.0.0.0 --port 8001 > server.log 2>&1 &

# Wait up to 120 seconds for server to start
import time, requests

for i in range(120):
    try:
        r = requests.get("http://localhost:8001/")
        print("Server is ready for Mary Ann!")
        break
    except:
        print(f"Waiting for server... {i+1}s")
        time.sleep(1)

# ------------------------------------------------------------
# 3. Test your API
# ------------------------------------------------------------

import io

image_url = "https://images.unsplash.com/photo-1543466835-00a7907e9de1?auto=format&fit=crop&w=1074&q=80"
image_response = requests.get(image_url)
image_bytes = io.BytesIO(image_response.content)

files = {"file": ("dog.jpg", image_bytes, "image/jpeg")}
response = requests.post("http://localhost:8001/detect/", files=files)

print("Status code:", response.status_code)
print(response.json())

# ------------------------------------------------------------
# 4. Stop your API
# ------------------------------------------------------------

!pkill uvicorn
print("Object detection API stopped for Mary Ann.")



Created object_detection_api.py for Mary Ann
Waiting for server... 1s
Waiting for server... 2s
Waiting for server... 3s
Waiting for server... 4s
Waiting for server... 5s
Server is ready for Mary Ann!
Status code: 200
{'filename': 'dog.jpg', 'detections': [{'label': 'dog', 'confidence': 0.9065, 'bounding_box': {'x1': 197.73, 'y1': 100.95, 'x2': 810.55, 'y2': 804.57}}]}
Object detection API stopped for Mary Ann.


## Reflective Questions

Please answer the following questions in a new Markdown cell below.

1.  Why is it better to deploy a model as an API instead of just having it as a script on your computer? What are the advantages?

Deploying a model as an API is better than keeping it as a local script because it allows anyone to send requests without needing the same environment or dependencies.   APIs also centralize updates and enable security features like authentication, logging and monitoring. Also, the cloud allows for scalability that local models just are incapable of doing.



2.  FastAPI automatically creates documentation for your API. How could this be useful for a team of developers working on a project?


FastAPI's automatic documentation is useful for teams because it clearly shows available endpoints, required inputs, and expected outputs. The documentation lets developers quickly understand the API's structure, test endpoints directly in the browser and get new team members up to speed without having them dig through source code. Documentation is meant to reduce miscommunication between the frontend and backend teams and make sure everyone is on the same page.  Also, APIs are able to authenticate, log, and monitor far easily than a local script can.  And because APIs are hosted in the cloud, you can easily scale which local scripts just cannot really do - not like in the cloud.

3.  Our API was deployed locally on our computer. What are some of the challenges you would face if you wanted to deploy this API to the cloud so that anyone in the world could use it? (Think about scalability, cost, security, etc.)

Cost of cloud compute is expensive.  Exposing the API publicly means you must have authentication, rate limiting and protection against abuse.  Scalability is good since the cloud can handle thousands of requests but then you have to also look at load balancing, autoscaling and monitoring. Maintenance is also centrally handled and can log, monitor and alert for any sort of suspicious activity or system failures.  Also, latency across different regions and data privacy issues must be considered when serving a global user base.

4.  What is MLOps and how does it relate to model deployment? Why is it an important field in AI?

Machine Learning operations covers the entire lifecycle:  data collection, training, deployment, retraining and scaling.  MLOps automates retraining and redeployment.  It bridges the gap between  the data scientists building the models and the engineers deploying and maintaining them.

## Submission Instructions

1.  Complete the **Student Challenge** section with your object detection API.
2.  Answer the **Reflective Questions** in a new Markdown cell.
3.  Save your completed notebook (`.ipynb` file) and submit it.